# Ceiling / Roof Segmentation Training на Kaggle

Этот ноутбук обучает `UNet + ResNet18` для бинарной сегментации **видимого потолка грузового отсека**.

Ожидаемые данные:
- `roof_selection.csv` с колонками `image_id`, `split` и желательно `load_bin`, `group_id`;
- `roof_masks/<image_id>.png`;
- `floor_images/<image_id>.*` — те же truck ROI, на которых делалась разметка;
- код проекта с `src/manual/roof_segmentation/{model.py,dataset.py,train.py,runtime.py}`.

Ноутбук сам ищет нужные пути внутри `/kaggle/input`, проверяет данные, запускает обучение на GPU и сохраняет лучший checkpoint в `/kaggle/working`.


In [ ]:
# 1. Проверка окружения и GPU
from pathlib import Path
import os
import sys
import json
import shutil
import subprocess

import torch
import torchvision
import pandas as pd
import numpy as np

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA недоступна. В Kaggle включи Settings -> Accelerator -> GPU."
    )

DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))

os.environ["TORCH_HOME"] = "/kaggle/working/torch_cache"
Path(os.environ["TORCH_HOME"]).mkdir(parents=True, exist_ok=True)


In [ ]:
# 2. Автоматический поиск актуального кода проекта
KAGGLE_INPUT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
WORK_SRC = WORK_ROOT / "src"

train_scripts = list(
    KAGGLE_INPUT.rglob("src/manual/roof_segmentation/train.py")
)

if not train_scripts:
    raise FileNotFoundError(
        "Не найден src/manual/roof_segmentation/train.py в /kaggle/input.\n"
        "Добавь Kaggle Dataset, содержащий актуальный код проекта."
    )

print("Найденные train.py:")
for p in train_scripts:
    print(" -", p)

TRAIN_SCRIPT_INPUT = train_scripts[0]
PROJECT_INPUT_ROOT = TRAIN_SCRIPT_INPUT.parents[3]
SRC_INPUT = PROJECT_INPUT_ROOT / "src"

required_code = [
    SRC_INPUT / "manual/roof_segmentation/model.py",
    SRC_INPUT / "manual/roof_segmentation/dataset.py",
    SRC_INPUT / "manual/roof_segmentation/train.py",
    SRC_INPUT / "manual/roof_segmentation/runtime.py",
]

missing_code = [str(p) for p in required_code if not p.exists()]
if missing_code:
    raise FileNotFoundError(
        "В найденном проекте отсутствуют файлы:\n" + "\n".join(missing_code)
    )

if WORK_SRC.exists():
    shutil.rmtree(WORK_SRC)
shutil.copytree(SRC_INPUT, WORK_SRC)

print("\nPROJECT_INPUT_ROOT:", PROJECT_INPUT_ROOT)
print("WORK_SRC:", WORK_SRC)
print("\nRoof segmentation files:")
for p in sorted((WORK_SRC / "manual/roof_segmentation").glob("*.py")):
    print(" -", p.name)


In [ ]:
# 3. Автоматический поиск roof_selection.csv, roof_masks и floor_images
selection_candidates = list(KAGGLE_INPUT.rglob("roof_selection.csv"))

if not selection_candidates:
    raise FileNotFoundError("Не найден roof_selection.csv в /kaggle/input.")

print("Найденные roof_selection.csv:")
for p in selection_candidates:
    print(" -", p)

valid_selection_candidates = []
for path in selection_candidates:
    try:
        tmp = pd.read_csv(path)
    except Exception:
        continue
    if {"image_id", "split"}.issubset(tmp.columns):
        valid_selection_candidates.append((len(tmp), path, tmp))

if not valid_selection_candidates:
    raise RuntimeError(
        "Ни один roof_selection.csv не содержит колонки image_id и split."
    )

valid_selection_candidates.sort(key=lambda x: x[0], reverse=True)
_, SELECTION_CSV, selection_df = valid_selection_candidates[0]
selection_df["image_id"] = selection_df["image_id"].astype(str)
selection_ids = selection_df["image_id"].tolist()

print("\nВыбран SELECTION_CSV:", SELECTION_CSV)
print("Строк:", len(selection_df))


def count_mask_matches(directory: Path, ids: list[str]) -> int:
    return sum((directory / f"{image_id}.png").exists() for image_id in ids)


mask_dirs = [p for p in KAGGLE_INPUT.rglob("roof_masks") if p.is_dir()]
if not mask_dirs:
    raise FileNotFoundError("Не найдена папка roof_masks в /kaggle/input.")

MASK_DIR = max(mask_dirs, key=lambda p: count_mask_matches(p, selection_ids))
mask_match_count = count_mask_matches(MASK_DIR, selection_ids)

print("Выбран MASK_DIR:", MASK_DIR)
print(f"Найдено масок для selection: {mask_match_count}/{len(selection_ids)}")

VALID_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]


def image_exists(directory: Path, image_id: str) -> bool:
    for ext in VALID_EXTENSIONS:
        if (directory / f"{image_id}{ext}").exists():
            return True
        if (directory / f"{image_id}{ext.upper()}").exists():
            return True
    return False


def count_image_matches(directory: Path, ids: list[str]) -> int:
    return sum(image_exists(directory, image_id) for image_id in ids)


image_dirs = [p for p in KAGGLE_INPUT.rglob("floor_images") if p.is_dir()]
if not image_dirs:
    raise FileNotFoundError(
        "Не найдена папка floor_images в /kaggle/input.\n"
        "Если truck ROI лежат под другим именем, измени IMAGE_DIR вручную в этой ячейке."
    )

IMAGE_DIR = max(image_dirs, key=lambda p: count_image_matches(p, selection_ids))
image_match_count = count_image_matches(IMAGE_DIR, selection_ids)

print("Выбран IMAGE_DIR:", IMAGE_DIR)
print(f"Найдено изображений для selection: {image_match_count}/{len(selection_ids)}")

if mask_match_count == 0:
    raise RuntimeError("Для выбранного roof_selection.csv не найдено ни одной маски.")
if image_match_count == 0:
    raise RuntimeError("Для выбранного roof_selection.csv не найдено ни одного изображения.")


In [ ]:
# 4. Проверка selection, split и целостности данных
df = pd.read_csv(SELECTION_CSV)
df["image_id"] = df["image_id"].astype(str)
df["split"] = df["split"].astype(str).str.lower()

print("Размер selection:", df.shape)
print("\nSplit:")
print(df["split"].value_counts(dropna=False))

if "load_bin" in df.columns:
    print("\nРаспределение load_bin:")
    print(
        df.groupby(["split", "load_bin"], observed=True)
          .size()
          .unstack(fill_value=0)
    )

train_ids = df[df["split"] == "train"]["image_id"].tolist()
val_ids = df[df["split"].isin(["validation", "val"])]["image_id"].tolist()

if not train_ids:
    raise RuntimeError("Нет train объектов.")
if not val_ids:
    raise RuntimeError("Нет validation объектов.")

id_overlap = set(train_ids) & set(val_ids)
print("\nTrain:", len(train_ids))
print("Validation:", len(val_ids))
print("Пересечение image_id:", len(id_overlap))

if id_overlap:
    raise RuntimeError(
        f"Есть пересечение image_id train/validation: {list(id_overlap)[:10]}"
    )

if "group_id" in df.columns:
    train_groups = set(df[df["split"] == "train"]["group_id"].astype(str))
    val_groups = set(
        df[df["split"].isin(["validation", "val"])]["group_id"].astype(str)
    )
    group_overlap = train_groups & val_groups
    print("Пересечение group_id:", len(group_overlap))
    if group_overlap:
        raise RuntimeError(
            "Есть leakage по group_id между train и validation. "
            f"Примеры: {list(group_overlap)[:10]}"
        )

missing_masks = [
    image_id for image_id in df["image_id"]
    if not (MASK_DIR / f"{image_id}.png").exists()
]
missing_images = [
    image_id for image_id in df["image_id"]
    if not image_exists(IMAGE_DIR, image_id)
]

print("\nНе найдено masks:", len(missing_masks))
print("Не найдено images:", len(missing_images))

if missing_masks:
    print("Примеры missing masks:", missing_masks[:10])
if missing_images:
    print("Примеры missing images:", missing_images[:10])

if missing_masks or missing_images:
    raise RuntimeError(
        "Датасет неполный. Исправь пути/данные до запуска обучения."
    )

if len(df) != 400:
    print(
        f"\nWARNING: selection содержит {len(df)} изображений, а ожидалось 400. "
        "Обучение возможно, но проверь, что это намеренно."
    )
else:
    print("\nOK: найдено ровно 400 размеченных изображений.")


In [ ]:
# 5. Импорт кода и smoke test dataset.py
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

from src.manual.roof_segmentation.dataset import (
    CeilingSegmentationDataset,
    validate_pairs,
)

validate_pairs(train_ids, IMAGE_DIR, MASK_DIR)
validate_pairs(val_ids, IMAGE_DIR, MASK_DIR)

train_dataset = CeilingSegmentationDataset(
    image_ids=train_ids,
    image_dir=IMAGE_DIR,
    mask_dir=MASK_DIR,
    image_size=320,
    augment=True,
    normalize=True,
)

val_dataset = CeilingSegmentationDataset(
    image_ids=val_ids,
    image_dir=IMAGE_DIR,
    mask_dir=MASK_DIR,
    image_size=320,
    augment=False,
    normalize=True,
)

sample = train_dataset[0]

print("image_id:", sample["image_id"])
print("image shape:", tuple(sample["image"].shape), sample["image"].dtype)
print("mask shape:", tuple(sample["mask"].shape), sample["mask"].dtype)
print("mask range:", float(sample["mask"].min()), float(sample["mask"].max()))
print("ceiling pixels:", int(sample["mask"].sum()))

assert tuple(sample["image"].shape) == (3, 320, 320)
assert tuple(sample["mask"].shape) == (1, 320, 320)

print("\nDATASET TEST PASSED")


In [ ]:
# 6. Smoke test model.py
from src.manual.roof_segmentation.model import build_model

model_test = build_model(pretrained=False).to(DEVICE)
model_test.eval()

x = torch.randn(2, 3, 320, 320, device=DEVICE)
with torch.no_grad():
    y = model_test(x)

print("Input:", tuple(x.shape))
print("Output logits:", tuple(y.shape))
assert tuple(y.shape) == (2, 1, 320, 320)

del model_test, x, y
torch.cuda.empty_cache()
print("\nMODEL TEST PASSED")


In [ ]:
# 7. Визуальная проверка нескольких image/mask пар
import matplotlib.pyplot as plt
from PIL import Image


def find_image_path(directory: Path, image_id: str) -> Path:
    for ext in VALID_EXTENSIONS:
        p = directory / f"{image_id}{ext}"
        if p.exists():
            return p
        p2 = directory / f"{image_id}{ext.upper()}"
        if p2.exists():
            return p2
    raise FileNotFoundError(image_id)


preview_ids = (
    df.sample(n=min(4, len(df)), random_state=42)["image_id"]
      .astype(str)
      .tolist()
)

fig, axes = plt.subplots(len(preview_ids), 3, figsize=(14, 4 * len(preview_ids)))
if len(preview_ids) == 1:
    axes = np.expand_dims(axes, axis=0)

for row_idx, image_id in enumerate(preview_ids):
    image = np.asarray(
        Image.open(find_image_path(IMAGE_DIR, image_id)).convert("RGB")
    )
    mask = np.asarray(
        Image.open(MASK_DIR / f"{image_id}.png").convert("L")
    )

    overlay = image.copy()
    roof = mask > 127
    overlay[roof] = (
        0.65 * overlay[roof] + 0.35 * np.array([0, 255, 0])
    ).astype(np.uint8)

    axes[row_idx, 0].imshow(image)
    axes[row_idx, 0].set_title(f"{image_id} | image")
    axes[row_idx, 0].axis("off")

    axes[row_idx, 1].imshow(mask, cmap="gray")
    axes[row_idx, 1].set_title("roof mask")
    axes[row_idx, 1].axis("off")

    axes[row_idx, 2].imshow(overlay)
    axes[row_idx, 2].set_title("overlay")
    axes[row_idx, 2].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# 8. Настройка обучения
ROOF_TRAIN_SCRIPT = WORK_SRC / "manual" / "roof_segmentation" / "train.py"

OUTPUT_DIR = WORK_ROOT / "roof_models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_IOU_PATH = OUTPUT_DIR / "ceiling_unet_resnet18.best_iou.pt"
BEST_LOSS_PATH = OUTPUT_DIR / "ceiling_unet_resnet18.best_loss.pt"
HISTORY_PATH = OUTPUT_DIR / "ceiling_unet_resnet18.history.json"

EPOCHS = 30
BATCH_SIZE = 16
NUM_WORKERS = 2

print("ROOF_TRAIN_SCRIPT:", ROOF_TRAIN_SCRIPT)
print("IMAGE_DIR:", IMAGE_DIR)
print("MASK_DIR:", MASK_DIR)
print("SELECTION_CSV:", SELECTION_CSV)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("EPOCHS:", EPOCHS)
print("BATCH_SIZE:", BATCH_SIZE)


In [ ]:
# 9. Запуск обучения CeilingUNet
train_command = [
    sys.executable,
    "-u",
    str(ROOF_TRAIN_SCRIPT),
    "--image-dir", str(IMAGE_DIR),
    "--mask-dir", str(MASK_DIR),
    "--selection-csv", str(SELECTION_CSV),
    "--output-dir", str(OUTPUT_DIR),
    "--device", DEVICE,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--image-size", "320",
    "--encoder-lr", "1e-4",
    "--decoder-lr", "3e-4",
    "--weight-decay", "1e-4",
    "--freeze-epochs", "2",
    "--patience", "7",
]

train_environment = os.environ.copy()
train_environment["PYTHONPATH"] = str(WORK_ROOT) + os.pathsep + train_environment.get("PYTHONPATH", "")

print("Команда:\n")
print(" ".join(train_command))
print("\nНачинаем обучение...\n")

result = subprocess.run(
    train_command,
    cwd=str(WORK_ROOT),
    env=train_environment,
    check=False,
)

if result.returncode != 0:
    raise RuntimeError(f"train.py завершился с кодом {result.returncode}")

print("\nTRAINING PROCESS FINISHED")


In [ ]:
# 10. Проверка артефактов обучения
print("Содержимое OUTPUT_DIR:")
for p in sorted(OUTPUT_DIR.glob("*")):
    size_mb = p.stat().st_size / (1024 ** 2)
    print(f" - {p.name}: {size_mb:.2f} MB")

required_outputs = [BEST_IOU_PATH, BEST_LOSS_PATH, HISTORY_PATH]
missing_outputs = [p for p in required_outputs if not p.exists()]

if missing_outputs:
    raise FileNotFoundError(
        "Не найдены ожидаемые артефакты:\n" + "\n".join(str(p) for p in missing_outputs)
    )

print("\nВсе ожидаемые артефакты найдены.")


In [ ]:
# 11. Метрики лучшего checkpoint
try:
    checkpoint = torch.load(
        BEST_IOU_PATH,
        map_location="cpu",
        weights_only=False,
    )
except TypeError:
    checkpoint = torch.load(BEST_IOU_PATH, map_location="cpu")

print("=== BEST IOU CHECKPOINT ===")
print("architecture:", checkpoint.get("architecture"))
print("epoch:", checkpoint.get("epoch"))
print("image_size:", checkpoint.get("image_size"))
print("threshold:", checkpoint.get("threshold"))
print("val_loss:", checkpoint.get("val_loss"))
print("val_iou:", checkpoint.get("val_iou"))
print("val_dice:", checkpoint.get("val_dice"))

with open(HISTORY_PATH, "r", encoding="utf-8") as f:
    history = json.load(f)

print("\nBest IoU from history:", history.get("best_iou"))
print("Best loss from history:", history.get("best_loss"))


In [ ]:
# 12. Runtime smoke test + визуализация prediction на validation
RUNTIME_SCRIPT = WORK_SRC / "manual" / "roof_segmentation" / "runtime.py"

if not RUNTIME_SCRIPT.is_file() or RUNTIME_SCRIPT.stat().st_size == 0:
    print(
        "Runtime smoke test пропущен: "
        "src/manual/roof_segmentation/runtime.py пустой или отсутствует.\n"
        "Checkpoint уже обучен и проверен на уровне артефактов; "
        "для inference сначала добавь реализацию CeilingSegmenter."
    )
else:
    from src.manual.roof_segmentation.runtime import CeilingSegmenter

    segmenter = CeilingSegmenter(
        weights_path=BEST_IOU_PATH,
        device="cuda",
    )

    example_id = val_ids[0]
    example_image = Image.open(
        find_image_path(IMAGE_DIR, example_id)
    ).convert("RGB")
    example_gt = np.asarray(
        Image.open(MASK_DIR / f"{example_id}.png").convert("L")
    ) > 127

    prediction = segmenter.predict(example_image)
    probability = prediction["probability"]
    pred_mask = prediction["mask"].astype(bool)
    intersection = np.logical_and(pred_mask, example_gt).sum()
    union = np.logical_or(pred_mask, example_gt).sum()
    example_iou = 1.0 if union == 0 else intersection / union

    print("image_id:", example_id)
    print("threshold:", prediction["threshold"])
    print("prediction shape:", pred_mask.shape)
    print("example IoU:", example_iou)

    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    axes[0].imshow(example_image)
    axes[0].set_title("Truck ROI")
    axes[0].axis("off")
    axes[1].imshow(example_gt, cmap="gray")
    axes[1].set_title("Ground truth")
    axes[1].axis("off")
    axes[2].imshow(probability, cmap="viridis", vmin=0, vmax=1)
    axes[2].set_title("Ceiling probability")
    axes[2].axis("off")
    axes[3].imshow(pred_mask, cmap="gray")
    axes[3].set_title(f"Prediction | IoU={example_iou:.3f}")
    axes[3].axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
# 13. Подготовка финальных файлов в /kaggle/working
FINAL_MODEL = WORK_ROOT / "ceiling_unet_resnet18.best_iou.pt"
FINAL_HISTORY = WORK_ROOT / "ceiling_unet_resnet18.history.json"

shutil.copy2(BEST_IOU_PATH, FINAL_MODEL)
shutil.copy2(HISTORY_PATH, FINAL_HISTORY)

ARCHIVE_PATH = WORK_ROOT / "ceiling_segmentation_artifacts.zip"
if ARCHIVE_PATH.exists():
    ARCHIVE_PATH.unlink()

import zipfile
with zipfile.ZipFile(
    ARCHIVE_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(BEST_IOU_PATH, arcname=BEST_IOU_PATH.name)
    archive.write(BEST_LOSS_PATH, arcname=BEST_LOSS_PATH.name)
    archive.write(HISTORY_PATH, arcname=HISTORY_PATH.name)

print("Готово.")
print("Best model:", FINAL_MODEL)
print("History:", FINAL_HISTORY)
print("Archive:", ARCHIVE_PATH)
print("\nПосле обучения сохрани Kaggle Version, чтобы файлы из /kaggle/working попали в Outputs.")
